# 🇷🇼 Rwanda Environment, Population & Resources – GIS Group Assignment (GitHub edition)
**University of Rwanda · CST · School of Architecture and Built Environment · Dept of Spatial Planning**

This notebook reads **ready-made, clipped data files from our GitHub repository** and produces the maps, charts and tables for your report. **No Google Earth Engine, no sign-in, no software to install** – it runs in Google Colab on a phone.

### 📱 How to use
1. Run **Cell 1** (installs tools, ~1 min).
2. Run **Cell 2** (helper tools).
3. Run **Cell 3**: type the repository link given by your instructor and your group details.
4. Run the unit cells **one by one, top to bottom**. Each cell prints `📌 Figure n / Table n` – **mention every one in your text**.
5. Run the **last cell** → a **Word draft** (cover page, TOC, list of figures/tables, all outputs, references).

> ⚠️ Free global datasets are modelled / satellite-derived. **Cross-check with national sources** (NISR, REMA, RDB, RWB, MINALOC, MoE) and cite both.
> 🎓 Want to do the same for another district? Ask your instructor to add `data/<district>/` to the repo – the notebook works for any folder.

In [ ]:
#@title 1️⃣ Install tools (run once – about 1 minute)
!pip -q install rasterio python-docx mapclassify
import os, io, json, math, subprocess, shutil, warnings
import numpy as np, pandas as pd, geopandas as gpd, rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, LightSource, to_rgba
from matplotlib.patches import Patch
from rasterio.features import geometry_mask
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.dpi': 110, 'font.size': 9})
print('✅ Tools ready')

In [ ]:
#@title 2️⃣ Helper tools (run once – no changes needed)
#@markdown Tick to save results in your Google Drive (otherwise they stay in this Colab session and the Word file is downloaded at the end).
SAVE_TO_DRIVE = False #@param {type:"boolean"}
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT = '/content/drive/MyDrive/GIS_Group_Report'
else:
    OUT = '/content/GIS_Group_Report'
os.makedirs(OUT + '/figures', exist_ok=True)
os.makedirs(OUT + '/tables', exist_ok=True)

FIGS, TABS, SOURCES = [], [], {}
UNIT = 'Introduction'

def set_unit(u):
    global UNIT
    UNIT = u

def cite(key, ref):
    SOURCES[key] = ref

def save_fig(fig, caption):
    n = len(FIGS) + 1
    path = f'{OUT}/figures/Figure_{n:02d}.png'
    fig.savefig(path, dpi=200, bbox_inches='tight')
    FIGS.append(dict(n=n, unit=UNIT, caption=caption, path=path))
    plt.show()
    print(f'📌 Figure {n}: {caption}   <- refer to it in your text')

def show_table(df, caption):
    n = len(TABS) + 1
    path = f'{OUT}/tables/Table_{n:02d}.csv'
    df.to_csv(path, index=False)
    TABS.append(dict(n=n, unit=UNIT, caption=caption, path=path))
    print(f'📌 Table {n}: {caption}   <- refer to it in your text')
    display(df)

def chart(kind, x, y, title, xlabel, ylabel, caption, color='#2b8cbe', horizontal=False):
    fig, ax = plt.subplots(figsize=(6.5, 3.6))
    if kind == 'bar':
        (ax.barh if horizontal else ax.bar)(x, y, color=color)
    else:
        ax.plot(x, y, marker='o', color=color)
    ax.set_title(title, weight='bold')
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    fig.tight_layout()
    save_fig(fig, caption)

def load_raster(name):
    with rasterio.open(os.path.join(BASE, name)) as ds:
        a = ds.read(1).astype('float32')
        if ds.nodata is not None:
            a[a == ds.nodata] = np.nan
        return a, ds.transform, ds.crs

def cell_km2(tr):
    return abs(tr.a * tr.e) / 1e6

def extent_of(a, tr):
    return (tr.c, tr.c + tr.a * a.shape[1], tr.f + tr.e * a.shape[0], tr.f)

def hillshade_of(dem, tr):
    z = np.where(np.isnan(dem), np.nanmean(dem), dem)
    return LightSource(azdeg=315, altdeg=45).hillshade(z, vert_exag=1, dx=abs(tr.a), dy=abs(tr.e))

def raster_map(arr, tr, title, caption, cmap=None, vmin=None, vmax=None, label=None, classes=None, hill=None):
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    ext = extent_of(arr, tr)
    if hill is not None and hill.shape == arr.shape:
        ax.imshow(hill, cmap='gray', extent=ext, vmin=0, vmax=1)
        alpha = 0.7
    else:
        alpha = 1.0
    if classes:
        rgba = np.zeros(arr.shape + (4,))
        for cid, _, col in classes:
            rgba[arr == cid] = to_rgba('#' + col, alpha)
        ax.imshow(rgba, extent=ext)
        ax.legend(handles=[Patch(color='#' + c, label=n) for _, n, c in classes], loc='upper center',
                  bbox_to_anchor=(0.5, -0.01), ncol=3, fontsize=7, frameon=False)
    else:
        if isinstance(cmap, list):
            cmap = LinearSegmentedColormap.from_list('c', ['#' + c for c in cmap])
        im = ax.imshow(arr, cmap=cmap, vmin=vmin, vmax=vmax, extent=ext, alpha=alpha)
        cb = fig.colorbar(im, ax=ax, shrink=0.6, orientation='horizontal', pad=0.02)
        cb.set_label(label or '')
    BND.boundary.plot(ax=ax, color='k', linewidth=1)
    ax.set_axis_off()
    ax.set_title(title, fontsize=11, weight='bold')
    x0, x1, y0, y1 = ext
    ax.plot([x0 + 0.05 * (x1 - x0), x0 + 0.05 * (x1 - x0) + 5000], [y0 + 0.03 * (y1 - y0)] * 2, color='k', lw=3)
    ax.text(x0 + 0.05 * (x1 - x0) + 2500, y0 + 0.045 * (y1 - y0), '5 km', ha='center', fontsize=8)
    ax.annotate('N\n↑', xy=(0.95, 0.93), xycoords='axes fraction', ha='center', fontsize=12, weight='bold')
    save_fig(fig, caption)

def area_by_class(arr, names, cellkm2):
    v = arr[~np.isnan(arr)].astype(int)
    vals, cnt = np.unique(v, return_counts=True)
    df = pd.DataFrame({'cls': vals, 'Area (km²)': cnt * cellkm2})
    df['Class'] = [names.get(c, f'Class {c}') for c in df['cls']]
    df['Share (%)'] = (df['Area (km²)'] / df['Area (km²)'].sum() * 100).round(1)
    df['Area (km²)'] = df['Area (km²)'].round(1)
    return df[['Class', 'Area (km²)', 'Share (%)']].sort_values('Area (km²)', ascending=False).reset_index(drop=True)

def zonal_sum(arr, tr, gdf):
    out = []
    for geom in gdf.geometry:
        m = geometry_mask([geom], out_shape=arr.shape, transform=tr, invert=True)
        out.append(float(np.nansum(arr[m])))
    return out

print('✅ Helper tools loaded')

In [ ]:
#@title 3️⃣ Connect to the data on GitHub and set your group details
#@markdown Repository link from your instructor, e.g. https://github.com/INSTRUCTOR/rwanda-gis-data
REPO = "https://github.com/YOUR-USERNAME/YOUR-REPO" #@param {type:"string"}
#@markdown Folder with the study-area data inside the repo
FOLDER = "data/gicumbi" #@param {type:"string"}
GROUP_NUMBER = 1 #@param {type:"integer"}
PROGRAMME = "EPM" #@param ["URP","EPM"]
#@markdown Group members (separated by commas) – printed on the cover page
MEMBERS = "" #@param {type:"string"}

if REPO.startswith('http'):
    if 'YOUR-USERNAME' in REPO:
        raise ValueError('Replace REPO with the real repository link given by your instructor.')
    shutil.rmtree('/content/repo', ignore_errors=True)
    r = subprocess.run(['git', 'clone', '--depth', '1', REPO, '/content/repo'], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError('Could not clone the repository (is it public?):\n' + r.stderr[-300:])
    BASE = os.path.join('/content/repo', FOLDER)
else:
    BASE = os.path.join(REPO, FOLDER)

meta = json.load(open(os.path.join(BASE, 'meta.json')))
STUDY_NAME = meta['study_name']
YEAR_FROM, YEAR_TO = meta['year_from'], meta['year_to']
dem, TR, CRS = load_raster('dem.tif')
CELL = cell_km2(TR)
dist_gdf = gpd.read_file(os.path.join(BASE, 'boundary_district.geojson'))
sects = gpd.read_file(os.path.join(BASE, 'boundary_sectors.geojson'))
rw = gpd.read_file(os.path.join(BASE, 'rwanda_districts.geojson'))
BND = dist_gdf.to_crs(CRS)
SECT = sects.to_crs(CRS)
AREA_KM2 = float(BND.area.iloc[0] / 1e6)
print(f'✅ Study area: {STUDY_NAME} | area ≈ {AREA_KM2:,.0f} km² | sectors: {len(sects)} | data exported {meta["exported"]}')

set_unit('Introduction')
cite('geoBoundaries', 'Runfola, D., Anderson, A., Baier, H., Crittenden, M., Dowker, E., Fuhrig, S., et al. (2020). geoBoundaries: A global database of political administrative boundaries. PLoS ONE, 15(4), e0231866. https://doi.org/10.1371/journal.pone.0231866')
fig, ax = plt.subplots(figsize=(6.5, 6))
rw.plot(ax=ax, color='#eeeeee', edgecolor='#999999', linewidth=0.5)
dist_gdf.plot(ax=ax, color='#9ecae1', edgecolor='#08519c', linewidth=1.5)
sects.boundary.plot(ax=ax, color='#08519c', linewidth=0.4)
ax.set_title(f'Location of {STUDY_NAME} District in Rwanda', weight='bold'); ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude')
save_fig(fig, f'Location of {STUDY_NAME} District within Rwanda, with its sectors.')

---
# 🌄 UNIT 1 – Physical Features

In [ ]:
#@title 🌄 1A – Relief: elevation and slope
set_unit('Unit 1 – Physical Features')
cite('SRTM', 'NASA JPL. (2013). NASA Shuttle Radar Topography Mission Global 1 arc second [Data set]. NASA EOSDIS Land Processes DAAC. https://doi.org/10.5067/MEaSUREs/SRTM/SRTMGL1.003')
HILL = hillshade_of(dem, TR)
emin, emax, emean, estd = np.nanmin(dem), np.nanmax(dem), np.nanmean(dem), np.nanstd(dem)
show_table(pd.DataFrame({'Indicator': ['Minimum elevation (m)', 'Maximum elevation (m)', 'Mean elevation (m)', 'Std. deviation (m)'],
                         'Value': [round(emin), round(emax), round(emean), round(estd)]}),
           f'Summary statistics of elevation in {STUDY_NAME} (SRTM 30 m).')
raster_map(dem, TR, f'Elevation – {STUDY_NAME}', f'Elevation map of {STUDY_NAME} (SRTM 30 m).',
           cmap=['006837', '78c679', 'ffffbf', 'd9a066', '8c510a', 'ffffff'], vmin=emin, vmax=emax, label='Elevation (m a.s.l.)', hill=HILL)

fig, ax = plt.subplots(figsize=(6.5, 3.4))
ax.hist(dem[~np.isnan(dem)], bins=30, color='#8c510a', edgecolor='white')
ax.set_title('Distribution of elevation', weight='bold'); ax.set_xlabel('Elevation (m)'); ax.set_ylabel('Number of cells'); ax.grid(alpha=0.3)
save_fig(fig, f'Histogram of elevation in {STUDY_NAME}.')

step = 300
edges = list(np.arange(np.floor(emin / step) * step + step, emax, step))
ecls = np.where(np.isnan(dem), np.nan, np.digitize(dem, edges) + 1)
labels = {}
lo = np.floor(emin / step) * step
bounds = [lo] + edges + [np.ceil(emax)]
for i in range(len(bounds) - 1):
    labels[i + 1] = f'{int(bounds[i])}–{int(bounds[i + 1])} m'
show_table(area_by_class(ecls, labels, CELL), f'Area by elevation class in {STUDY_NAME}.')

dzdy, dzdx = np.gradient(np.where(np.isnan(dem), np.nanmean(dem), dem), abs(TR.a))
slope_pct = np.hypot(dzdx, dzdy) * 100
bad = np.isnan(dem)
edge = bad.copy()
for dy in (-1, 0, 1):
    for dx in (-1, 0, 1):
        edge |= np.roll(np.roll(bad, dy, 0), dx, 1)
slope_pct[edge] = np.nan
scls = np.where(np.isnan(slope_pct), np.nan, np.digitize(slope_pct, [6, 16, 30, 55]) + 1)
sn = {1: 'Flat–gentle (<6%)', 2: 'Moderate (6–16%)', 3: 'Strong (16–30%)', 4: 'Steep (30–55%)', 5: 'Very steep (>55%)'}
scol = ['1a9850', 'a6d96a', 'fee08b', 'f46d43', 'a50026']
raster_map(scls, TR, f'Slope classes – {STUDY_NAME}', f'Slope classes (%) of {STUDY_NAME}, derived from SRTM.',
           classes=[(i + 1, sn[i + 1], scol[i]) for i in range(5)], hill=HILL)
show_table(area_by_class(scls, sn, CELL), f'Area by slope class in {STUDY_NAME}.')

✍️ **Write-up guide (Unit 1A):** Describe the relief. Which elevation zone holds most land? How do steep slopes affect **farming (terracing), settlement location, roads, landslide/erosion risk**?

In [ ]:
#@title 🌦️ 1B – Climate: rainfall and temperature
set_unit('Unit 1 – Physical Features')
cite('CHIRPS', 'Funk, C., Peterson, P., Landsfeld, M., Pedreros, D., Verdin, J., Shukla, S., et al. (2015). The climate hazards infrared precipitation with stations - a new environmental record for monitoring extremes. Scientific Data, 2, 150066. https://doi.org/10.1038/sdata.2015.66')
cite('ERA5-Land', 'Muñoz Sabater, J. (2019). ERA5-Land monthly averaged data from 1950 to present. Copernicus Climate Change Service (C3S) Climate Data Store. https://doi.org/10.24381/cds.68d2bb30')
mdf = pd.read_csv(os.path.join(BASE, 'climate_monthly.csv')).sort_values('month')
ydf = pd.read_csv(os.path.join(BASE, 'rain_annual.csv')).sort_values('year')
mn = ['Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec']
mdf['Month'] = mn
tab = mdf[['Month', 'rain', 'temp']].round(1).rename(columns={'rain': 'Rainfall (mm)', 'temp': 'Mean temperature (°C)'})
show_table(tab, f'Monthly mean rainfall and temperature, {STUDY_NAME}, {YEAR_FROM}–{YEAR_TO}.')

fig, ax = plt.subplots(figsize=(6.5, 3.8))
ax.bar(mn, mdf['rain'], color='#4292c6')
ax.set_ylabel('Rainfall (mm/month)')
ax2 = ax.twinx()
ax2.plot(mn, mdf['temp'], color='#d94801', marker='o')
ax2.set_ylabel('Temperature (°C)')
ax.set_title(f'Climate diagram – {STUDY_NAME}', weight='bold')
fig.tight_layout()
save_fig(fig, f'Climate diagram of {STUDY_NAME} (CHIRPS rainfall, ERA5-Land temperature, {YEAR_FROM}–{YEAR_TO}).')

fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.bar(ydf['year'], ydf['rain'], color='#6baed6')
ax.axhline(ydf['rain'].mean(), color='k', ls='--', lw=1, label=f"Mean = {ydf['rain'].mean():.0f} mm")
ax.set_title('Annual rainfall', weight='bold'); ax.set_xlabel('Year'); ax.set_ylabel('mm/year'); ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, f'Annual rainfall totals in {STUDY_NAME}, {YEAR_FROM}–{YEAR_TO} (CHIRPS).')
wet = mdf.loc[mdf['rain'].idxmax(), 'Month']; dry = mdf.loc[mdf['rain'].idxmin(), 'Month']
print(f"Mean annual rainfall ≈ {ydf['rain'].mean():.0f} mm | wettest month: {wet} | driest month: {dry} | mean temperature ≈ {mdf['temp'].mean():.1f} °C")

✍️ **Write-up guide (Unit 1B):** Identify the **rainy seasons (Mar–May, Sep–Dec)** and dry seasons. How do rainfall/temperature affect crops, planting calendar, flood/drought risk, settlement? Any trend in annual rainfall?

In [ ]:
#@title 🪨 1C – Soils (texture and pH)
set_unit('Unit 1 – Physical Features')
cite('OpenLandMap', 'Hengl, T., et al. (2018). OpenLandMap soil texture class (USDA system) and soil pH in water at 250 m (v02) [Data set]. OpenLandMap. https://openlandmap.org')
tex, ttr, _ = load_raster('soil_texture.tif')
tn = {1: 'Clay', 2: 'Silty clay', 3: 'Sandy clay', 4: 'Clay loam', 5: 'Silty clay loam', 6: 'Sandy clay loam',
      7: 'Loam', 8: 'Silt loam', 9: 'Sandy loam', 10: 'Silt', 11: 'Loamy sand', 12: 'Sand'}
tcol = ['8c2d04', 'cc4c02', 'ec7014', 'fe9929', 'fec44f', 'fee391', 'a6bddb', '74a9cf', 'fff7bc', 'd9f0a3', 'addd8e', '78c679']
tdf = area_by_class(tex, tn, cell_km2(ttr))
raster_map(tex, ttr, f'Topsoil texture – {STUDY_NAME}', f'Dominant topsoil texture class (0 cm depth) in {STUDY_NAME}.',
           classes=[(i + 1, tn[i + 1], tcol[i]) for i in range(12) if tn[i + 1] in set(tdf['Class'])])
show_table(tdf, f'Area by topsoil texture class in {STUDY_NAME}.')
ph, ptr, _ = load_raster('soil_ph_x10.tif')
ph = ph / 10
raster_map(ph, ptr, f'Topsoil pH – {STUDY_NAME}', f'Topsoil pH (in water) in {STUDY_NAME}.',
           cmap=['d73027', 'fc8d59', 'fee08b', 'd9ef8b', '66bd63', '1a9850'], vmin=4.5, vmax=7.5, label='pH')
print('Mean topsoil pH ≈', round(float(np.nanmean(ph)), 2), '(<5.5 = strongly acidic: lime needed for many crops)')
print('⚠️ Modelled global soil maps at 250 m – validate with RAB / MINAGRI soil survey data in your text.')

In [ ]:
#@title 🌳 1D – Vegetation / land cover (ESA WorldCover 10 m, resampled to 30 m)
set_unit('Unit 1 – Physical Features')
cite('WorldCover', 'Zanaga, D., Van De Kerchove, R., Daems, D., De Keersmaecker, W., Brockmann, C., Kirches, G., et al. (2022). ESA WorldCover 10 m 2021 v200 [Data set]. Zenodo. https://doi.org/10.5281/zenodo.7254221')
lc, ltr, _ = load_raster('worldcover.tif')
LCN = {10: 'Tree cover', 20: 'Shrubland', 30: 'Grassland', 40: 'Cropland', 50: 'Built-up', 60: 'Bare / sparse', 70: 'Snow / ice',
       80: 'Permanent water', 90: 'Herbaceous wetland', 95: 'Mangroves', 100: 'Moss / lichen'}
LCC = {10: '006400', 20: 'ffbb22', 30: 'ffff4c', 40: 'f096ff', 50: 'fa0000', 60: 'b4b4b4', 70: 'f0f0f0', 80: '0064c8', 90: '0096a0', 95: '00cf75', 100: 'fae6a0'}
lcdf = area_by_class(lc, LCN, cell_km2(ltr))
present = [k for k, v in LCN.items() if v in set(lcdf['Class'])]
raster_map(lc, ltr, f'Land cover – {STUDY_NAME}', f'Land cover of {STUDY_NAME} (ESA WorldCover 2021).',
           classes=[(k, LCN[k], LCC[k]) for k in present], hill=HILL)
show_table(lcdf, f'Land-cover composition of {STUDY_NAME} (ESA WorldCover 2021).')
fig, ax = plt.subplots(figsize=(5.5, 5))
big = lcdf[lcdf['Share (%)'] >= 1]
ax.pie(big['Area (km²)'], labels=big['Class'], autopct='%1.0f%%', startangle=90,
       colors=['#' + LCC[[k for k, v in LCN.items() if v == c][0]] for c in big['Class']], textprops={'fontsize': 8})
ax.set_title('Land-cover shares (classes ≥ 1%)', weight='bold')
save_fig(fig, f'Share of land-cover classes in {STUDY_NAME}.')

In [ ]:
#@title 💧 1E – Water resources (surface water)
set_unit('Unit 1 – Physical Features')
cite('JRC-GSW', 'Pekel, J.-F., Cottam, A., Gorelick, N., & Belward, A. S. (2016). High-resolution mapping of global surface water and its long-term changes. Nature, 540, 418-422. https://doi.org/10.1038/nature20584')
occ, otr, _ = load_raster('water_occurrence.tif')
occm = np.where(occ >= 5, occ, np.nan)
raster_map(occm, otr, f'Surface water occurrence – {STUDY_NAME}', f'Surface-water occurrence (1984–2021) in {STUDY_NAME}: rivers, wetlands and reservoirs.',
           cmap=['c6dbef', '6baed6', '2171b5', '08306b'], vmin=5, vmax=100, label='% of time with water', hill=HILL)
wcls = np.where(occ >= 50, 2, np.where(occ >= 10, 1, np.nan))
show_table(area_by_class(wcls, {1: 'Seasonal water (10–50% of time)', 2: 'Permanent water (≥ 50% of time)'}, cell_km2(otr)),
           f'Surface-water area in {STUDY_NAME} (JRC Global Surface Water).')

✍️ **Write-up guide (Unit 1C–E):** Dominant soils, vegetation/land-cover, rivers, wetlands and lakes (use RWB & REMA too). End Unit 1 by explaining **how physical features shape livelihoods, agriculture and settlement patterns**.

---
# 🏘️ UNIT 2 – Administrative Structure, Population & Human Settlements

In [ ]:
#@title 🗺️ 2A – Administrative divisions (sectors)
set_unit('Unit 2 – Administrative Structure, Population and Human Settlements')
t = SECT[['shapeName']].copy()
t['Area (km²)'] = (SECT.area / 1e6).round(1).values
t = t.rename(columns={'shapeName': 'Sector'}).sort_values('Area (km²)', ascending=False).reset_index(drop=True)
show_table(t, f'Sectors of {STUDY_NAME} District and their area.')
fig, ax = plt.subplots(figsize=(6.5, 6.5))
SECT.boundary.plot(ax=ax, color='#08519c', linewidth=0.8)
BND.boundary.plot(ax=ax, color='k', linewidth=1.8)
for _, r in SECT.iterrows():
    p = r.geometry.representative_point()
    ax.annotate(r['shapeName'], (p.x, p.y), ha='center', fontsize=6)
ax.set_axis_off(); ax.set_title(f'Administrative sectors – {STUDY_NAME}', weight='bold')
save_fig(fig, f'Sectors of {STUDY_NAME} District.')
print(f'ℹ️ {STUDY_NAME} has {len(t)} sectors. Cells and villages are not in the free dataset – get them from MINALOC / NISR and add them in your text.')

In [ ]:
#@title 👥 2B – Population: size, density, growth, distribution
set_unit('Unit 2 – Administrative Structure, Population and Human Settlements')
cite('WorldPop', 'WorldPop. (2018). Global high resolution population denominators project (Funded by The Bill and Melinda Gates Foundation, OPP1134076). University of Southampton. https://doi.org/10.5258/SOTON/WP00645')
ts = pd.read_csv(os.path.join(BASE, 'pop_timeseries.csv')).sort_values('year')
p0, p1 = ts['pop'].iloc[0], ts['pop'].iloc[-1]
cagr = (p1 / p0) ** (1 / (ts['year'].iloc[-1] - ts['year'].iloc[0])) - 1
chart('line', ts['year'], ts['pop'] / 1e3, f'Population trend – {STUDY_NAME}', 'Year', 'Population (thousands)',
      f'Modelled population of {STUDY_NAME}, 2000–2020 (WorldPop). Average annual growth ≈ {cagr * 100:.1f}%.', color='#cb181d')
print(f'Population 2020 ≈ {p1:,.0f}; density ≈ {p1 / AREA_KM2:,.0f} people/km²; average annual growth ≈ {cagr * 100:.2f}%')

pop, pr, _ = load_raster('pop_2020.tif')
u = SECT[['shapeName', 'geometry']].copy().reset_index(drop=True)
u['Population'] = np.round(zonal_sum(pop, pr, u), 0)
u['Area (km²)'] = u.area / 1e6
u['Density (people/km²)'] = (u['Population'] / u['Area (km²)']).round(0)
fig, ax = plt.subplots(figsize=(6.5, 6.5))
try:
    u.plot(column='Density (people/km²)', cmap='YlOrRd', scheme='quantiles', k=5, legend=True, ax=ax, edgecolor='white', linewidth=0.3,
           legend_kwds={'title': 'People per km²', 'fontsize': 7, 'loc': 'lower left'})
except Exception:
    u.plot(column='Density (people/km²)', cmap='YlOrRd', legend=True, ax=ax, edgecolor='white', linewidth=0.3)
ax.set_axis_off(); ax.set_title(f'Population density 2020 – {STUDY_NAME}', weight='bold')
save_fig(fig, f'Population density by sector in {STUDY_NAME}, 2020 (WorldPop).')
top = u.sort_values('Density (people/km²)', ascending=False).head(10)
chart('bar', top['shapeName'], top['Density (people/km²)'], 'Top 10 most densely populated sectors', 'People per km²', '',
      f'Ten most densely populated sectors in {STUDY_NAME}, 2020.', color='#e6550d', horizontal=True)
show_table(top[['shapeName', 'Population', 'Area (km²)', 'Density (people/km²)']].round(1).rename(columns={'shapeName': 'Sector'}),
           f'Population and density of the ten most densely populated sectors in {STUDY_NAME}, 2020.')
print('⚠️ WorldPop is a model. Compare with the 2012 and 2022 Population & Housing Census figures from NISR and cite NISR as the official source.')

In [ ]:
#@title 🏙️ 2C – Human settlements: built-up growth (urbanization)
set_unit('Unit 2 – Administrative Structure, Population and Human Settlements')
cite('GHSL', 'Pesaresi, M., & Politis, P. (2023). GHS-BUILT-S R2023A - GHS built-up surface grid, derived from Sentinel2 composite and Landsat, multitemporal (1975-2030). European Commission, Joint Research Centre (JRC). https://doi.org/10.2905/9F06F36F-4B11-47EC-ABB0-4F8B7B1D72EA')
bt = pd.read_csv(os.path.join(BASE, 'built_timeseries.csv')).sort_values('year')
bt['Built-up area (km²)'] = (bt['built_m2'] / 1e6).round(2)
chart('line', bt['year'], bt['Built-up area (km²)'], f'Built-up area growth – {STUDY_NAME}', 'Year', 'Built-up area (km²)',
      f'Growth of built-up surface in {STUDY_NAME}, 1975–2020 (GHSL).', color='#636363')
show_table(bt[['year', 'Built-up area (km²)']].rename(columns={'year': 'Year'}), f'Built-up surface in {STUDY_NAME} by epoch (GHSL).')
b20, btr, _ = load_raster('built_2020.tif')
raster_map(np.where(b20 > 0, b20, np.nan), btr, f'Built-up surface 2020 – {STUDY_NAME}',
           f'Built-up surface in {STUDY_NAME}, 2020 (m² per 100 m cell, GHSL): shows clustered vs scattered settlement.',
           cmap=['fdd49e', 'fc8d59', 'd7301f', '7f0000'], vmin=0, vmax=3000, label='Built-up m² per 100 m cell')

✍️ **Write-up guide (Unit 2):** (1) Hierarchy: Province → District → Sectors → Cells → Villages (MINALOC). (2) Population size, density, growth and where people concentrate. (3) Settlements: urbanization, rural–urban migration, **Imidugudu** (clustered vs scattered pattern in the built-up map).

---
# 💼 UNIT 3 – Socio-Economic Activities

In [ ]:
#@title 🌙 3A – Economic activity proxy: night-time lights
set_unit('Unit 3 – Socio-Economic Activities')
cite('VIIRS-NTL', 'Elvidge, C. D., Zhizhin, M., Ghosh, T., Hsu, F.-C., & Taneja, J. (2021). Annual time series of global VIIRS nighttime lights derived from monthly averages: 2012 to 2019. Remote Sensing, 13(5), 922. https://doi.org/10.3390/rs13050922')
nt = pd.read_csv(os.path.join(BASE, 'ntl_timeseries.csv')).sort_values('year')
chart('line', nt['year'], nt['rad'], f'Night-time lights – {STUDY_NAME}', 'Year', 'Mean radiance (nW/cm²/sr)',
      f'Trend of mean night-time light radiance in {STUDY_NAME} (VIIRS) – a proxy for electrification and economic activity.', color='#6a51a3')
nl, ntr, _ = load_raster('ntl_latest.tif')
raster_map(np.log1p(np.clip(nl, 0, None)), ntr, f'Night-time lights (latest year) – {STUDY_NAME}',
           f'Night-time lights in {STUDY_NAME}, latest VIIRS year: bright areas = towns, industry, electrified settlements.',
           cmap=['3f007d', '807dba', 'fec44f', 'ffffff'], vmin=0, vmax=3.5, label='log(radiance + 1)')

In [ ]:
#@title 🏭 3B – Economic features (tourism, mining, industry, markets, services – OpenStreetMap)
set_unit('Unit 3 – Socio-Economic Activities')
cite('OSM', 'OpenStreetMap contributors. (2026). OpenStreetMap [Data set]. https://www.openstreetmap.org/copyright')
fp = os.path.join(BASE, 'osm_features.geojson')
if os.path.exists(fp):
    feats = gpd.read_file(fp)
    cnt = feats['category'].value_counts().reset_index()
    cnt.columns = ['Economic category', 'Number of mapped features']
    show_table(cnt, f'Mapped economic features in {STUDY_NAME} (OpenStreetMap; incomplete – indicative only).')
    chart('bar', cnt['Economic category'], cnt['Number of mapped features'], 'Mapped economic features', '', 'Count',
          f'Number of mapped economic features by category in {STUDY_NAME} (OpenStreetMap).', color='#31a354')
    fig, ax = plt.subplots(figsize=(6.5, 6.5))
    dist_gdf.boundary.plot(ax=ax, color='k', linewidth=1)
    cols = {'Tourism': '#e41a1c', 'Mining / quarry': '#8c510a', 'Industry': '#377eb8', 'Trade / commerce': '#ff7f00', 'Finance': '#984ea3', 'Health': '#4daf4a', 'Education': '#a65628', 'Energy': '#f781bf'}
    for k, c in cols.items():
        s = feats[feats['category'] == k]
        if len(s):
            s.plot(ax=ax, color=c, markersize=10, label=k)
    ax.legend(fontsize=7); ax.set_title(f'Economic features – {STUDY_NAME}', weight='bold'); ax.set_xlabel('Longitude (°E)'); ax.set_ylabel('Latitude')
    save_fig(fig, f'Location of tourism, mining, industry, trade and service features in {STUDY_NAME} (OpenStreetMap).')
else:
    print('No OSM file in this study-area folder – skipped.')
print('⚠️ OSM is volunteer-mapped and incomplete: complement with RDB investment/tourism/mining data, NISR Establishment Census and Labour Force Survey.')

In [ ]:
#@title 📊 3C – Add OFFICIAL data (NISR / RDB / MINALOC) → automatic charts
#@markdown Step 1: run this cell with **MAKE_TEMPLATE** ticked to create a CSV template in your Drive. Fill it in Google Sheets (one row per value; e.g. Indicator = *Employment by sector*, Category = *Agriculture*, Value = *62*, Year = *2022*, Source = *NISR LFS 2022*). Step 2: untick it, run again and upload the CSV.
MAKE_TEMPLATE = True #@param {type:"boolean"}
set_unit('Unit 3 – Socio-Economic Activities')
if MAKE_TEMPLATE:
    t = pd.DataFrame({'Indicator': ['Employment by sector'] * 3, 'Category': ['Agriculture', 'Industry', 'Services'], 'Value': ['', '', ''], 'Year': [2022] * 3, 'Source': ['NISR'] * 3})
    t.to_csv(OUT + '/NISR_data_template.csv', index=False)
    print('✅ Template saved to Drive > GIS_Group_Report > NISR_data_template.csv (open with Google Sheets, fill, download as CSV).')
else:
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(list(up)[0])
    df['Value'] = pd.to_numeric(df['Value'], errors='coerce')
    for ind, sub in df.dropna(subset=['Value']).groupby('Indicator'):
        yrs = sorted(sub['Year'].unique())
        s = sub[sub['Year'] == yrs[-1]]
        chart('bar', s['Category'], s['Value'], str(ind), '', 'Value', f'{ind}, {yrs[-1]} (source: {", ".join(s["Source"].astype(str).unique())}).', color='#2ca25f')
        show_table(sub[['Category', 'Value', 'Year', 'Source']], f'{ind} (official data).')
        for srcname in sub['Source'].astype(str).unique():
            cite('official-' + srcname, f'{srcname}. (Year). Title of report. Publisher. URL  <- complete this reference in APA 7')

✍️ **Write-up guide (Unit 3):** Dominant sectors, **development initiatives & investment zones (RDB, SEZ, industrial parks)**, employment patterns (NISR LFS), and challenges vs opportunities for inclusive development.

---
# 🌱 UNIT 4 – Environmental Activities

In [ ]:
#@title 🌲 4A – Forest cover and tree-cover loss
set_unit('Unit 4 – Environmental Activities')
cite('Hansen', 'Hansen, M. C., Potapov, P. V., Moore, R., Hancher, M., Turubanova, S. A., Tyukavina, A., et al. (2013). High-resolution global maps of 21st-century forest cover change. Science, 342(6160), 850-853. https://doi.org/10.1126/science.1244693')
tc, htr, _ = load_raster('treecover2000.tif')
ly, _, _ = load_raster('lossyear.tif')
tcm = tc >= 30
lossm = (ly > 0) & tcm
ha = cell_km2(htr) * 100
tc_area = np.nansum(tcm) * ha
rows = [(2000 + k, round(float(((ly == k) & tcm).sum() * ha), 1)) for k in range(1, int(np.nanmax(ly)) + 1)]
ldf = pd.DataFrame(rows, columns=['Year', 'Tree-cover loss (ha)'])
print(f'Tree cover in 2000 (>=30% canopy): {tc_area:,.0f} ha  |  Total loss since 2001: {ldf["Tree-cover loss (ha)"].sum():,.0f} ha')
chart('bar', ldf['Year'], ldf['Tree-cover loss (ha)'], f'Annual tree-cover loss – {STUDY_NAME}', 'Year', 'Hectares',
      f'Annual tree-cover loss in {STUDY_NAME} (Hansen et al.). Includes plantation harvesting – interpret with REMA / MINIRENA forestry data.', color='#b30000')
show_table(ldf, f'Annual tree-cover loss in {STUDY_NAME}.')
fc = np.where(lossm, 2, np.where(tcm, 1, np.nan))
raster_map(fc, htr, f'Tree cover and loss – {STUDY_NAME}', f'Tree cover in 2000 (green) and tree-cover loss 2001–latest (red) in {STUDY_NAME}.',
           classes=[(1, 'Tree cover 2000 (≥30%)', '238b45'), (2, 'Tree-cover loss', 'e31a1c')], hill=HILL)

In [ ]:
#@title ⛰️ 4B – Soil-erosion hot-spots: cultivated steep slopes
set_unit('Unit 4 – Environmental Activities')
if lc.shape != slope_pct.shape:
    raise ValueError('Land cover and DEM grids differ – re-export the study area with the instructor notebook.')
crop = lc == 40
hot = crop & (slope_pct >= 30)
a_crop, a_hot = crop.sum() * CELL, hot.sum() * CELL
show_table(pd.DataFrame({'Indicator': ['Cropland (km²)', 'Cropland on slopes ≥ 30% (km²)', 'Share of cropland on steep slopes (%)'],
                         'Value': [round(float(a_crop), 1), round(float(a_hot), 1), round(float(a_hot / a_crop * 100), 1) if a_crop else 0]}),
           f'Cropland on steep slopes in {STUDY_NAME} – priority for terracing, agroforestry and erosion control.')
cl = np.where(hot, 2, np.where(crop, 1, np.nan))
raster_map(cl, TR, f'Erosion-prone cropland – {STUDY_NAME}', f'Cropland and cropland on slopes ≥ 30% in {STUDY_NAME}: indicative soil-erosion hot-spots.',
           classes=[(1, 'Cropland', 'fdd49e'), (2, 'Cropland on slope ≥ 30%', 'b30000')], hill=HILL)

In [ ]:
#@title 🌿 4C – Vegetation health trend (NDVI)
set_unit('Unit 4 – Environmental Activities')
cite('MODIS-NDVI', 'Didan, K. (2021). MODIS/Terra Vegetation Indices 16-Day L3 Global 250m SIN Grid V061 [Data set]. NASA EOSDIS Land Processes DAAC. https://doi.org/10.5067/MODIS/MOD13Q1.061')
ndf = pd.read_csv(os.path.join(BASE, 'ndvi_timeseries.csv')).dropna().sort_values('year')
fit = np.polyfit(ndf['year'], ndf['ndvi'], 1)
fig, ax = plt.subplots(figsize=(6.5, 3.6))
ax.plot(ndf['year'], ndf['ndvi'], marker='o', color='#238b45', label='Annual mean NDVI')
ax.plot(ndf['year'], np.polyval(fit, ndf['year']), 'k--', label=f'Trend: {fit[0]:+.4f}/yr')
ax.set_title(f'NDVI trend – {STUDY_NAME}', weight='bold'); ax.set_xlabel('Year'); ax.set_ylabel('NDVI'); ax.legend(); ax.grid(alpha=0.3)
save_fig(fig, f'Annual mean NDVI in {STUDY_NAME} (MODIS) with linear trend.')
print('Rising NDVI = greening (reforestation, agroforestry); falling = possible degradation. Link to national programmes in your discussion.')

In [ ]:
#@title 🏛️ 4D – Link your findings to national strategies (EDIT the text below, then run)
set_unit('Unit 4 – Environmental Activities')
rows = [
    ['Reforestation / agroforestry programmes in my area', 'Vision 2050 / NST1-NST2 / Green Growth and Climate Resilience Strategy – explain the link', 'Source (e.g. REMA, MoE)'],
    ['Soil & water conservation (terracing, check dams)', 'TO FILL', 'TO FILL'],
    ['Climate adaptation (early warning, resilient agriculture)', 'TO FILL', 'TO FILL'],
    ['Waste management / wetland protection', 'TO FILL', 'TO FILL'],
    ['Key environmental challenge 1 (e.g. erosion, deforestation, flooding, pollution)', 'TO FILL', 'TO FILL'],
]
show_table(pd.DataFrame(rows, columns=['Programme or challenge', 'Link to national strategy', 'Source']), f'Environmental programmes and challenges in {STUDY_NAME} and their link to national strategies.')

✍️ **Write-up guide (Unit 4):** Describe real programmes in your area (reforestation, terracing, climate adaptation, waste management – ask the District / REMA / RWB). Use **forest loss, erosion hot-spots and NDVI** as evidence of challenges. Link explicitly to **Vision 2050, NST1/NST2 and the Green Growth and Climate Resilience Strategy**.

---
# 📄 Build your Word report draft

In [ ]:
#@title 📄 FINAL – Create the Word draft (cover page, TOC, list of figures/tables, all outputs, references)
DOWNLOAD_TO_PHONE = True #@param {type:"boolean"}
from docx import Document
from docx.shared import Pt, Cm
from docx.enum.text import WD_ALIGN_PARAGRAPH, WD_BREAK
from docx.oxml.ns import qn
from docx.oxml import OxmlElement

FONT = 'Segoe UI Emoji'
doc = Document()
for s in doc.sections:
    s.left_margin = s.right_margin = Cm(2.5)
    s.top_margin = s.bottom_margin = Cm(2.5)
def setfont(style, size, bold=False):
    style.font.name = FONT
    style.font.size = Pt(size)
    style.font.bold = bold
    style.element.rPr.rFonts.set(qn('w:eastAsia'), FONT)
    style.element.rPr.rFonts.set(qn('w:ascii'), FONT)
    style.element.rPr.rFonts.set(qn('w:hAnsi'), FONT)
setfont(doc.styles['Normal'], 12)
doc.styles['Normal'].paragraph_format.line_spacing = 1.5
for h, sz in [('Heading 1', 16), ('Heading 2', 14), ('Heading 3', 12)]:
    setfont(doc.styles[h], sz, True)
    doc.styles[h].font.color.rgb = None
setfont(doc.styles['Caption'], 10, True)

def center(text, size=12, bold=False):
    p = doc.add_paragraph()
    p.alignment = WD_ALIGN_PARAGRAPH.CENTER
    r = p.add_run(text); r.font.size = Pt(size); r.bold = bold
    return p

center('UNIVERSITY OF RWANDA', 16, True)
center('College of Science and Technology (CST)', 13, True)
center('School of Architecture and Built Environment – Department of Spatial Planning', 12)
for _ in range(3): doc.add_paragraph()
center(f'GROUP {GROUP_NUMBER} ({PROGRAMME} students)', 14, True)
center('Rwanda\'s Environment, Population, Economic Systems and Natural Resource Management', 14, True)
center(f'Study area: {STUDY_NAME}', 13, True)
for _ in range(2): doc.add_paragraph()
center('Group members', 12, True)
for m in [x.strip() for x in MEMBERS.split(',') if x.strip()] or ['(add names)']:
    center(m)
doc.add_paragraph().add_run().add_break(WD_BREAK.PAGE)

doc.add_heading('Table of Contents', 1)
p = doc.add_paragraph(); r = p.add_run()
for typ, txt in [('begin', None), ('instr', 'TOC \\o "1-3" \\h \\z \\u'), ('separate', None), ('text', 'Right-click here → Update field to build the table of contents.'), ('end', None)]:
    if typ in ('begin', 'separate', 'end'):
        e = OxmlElement('w:fldChar'); e.set(qn('w:fldCharType'), typ)
    elif typ == 'instr':
        e = OxmlElement('w:instrText'); e.set(qn('xml:space'), 'preserve'); e.text = txt
    else:
        e = OxmlElement('w:t'); e.text = txt
    r._r.append(e)
doc.add_paragraph().add_run().add_break(WD_BREAK.PAGE)

doc.add_heading('List of Tables', 1)
for t in TABS: doc.add_paragraph(f'Table {t["n"]}: {t["caption"]}')
doc.add_heading('List of Figures', 1)
for f in FIGS: doc.add_paragraph(f'Figure {f["n"]}: {f["caption"]}')
doc.add_paragraph().add_run().add_break(WD_BREAK.PAGE)

def add_table(t):
    df = pd.read_csv(t['path']).head(40)
    cap = doc.add_paragraph(f'Table {t["n"]}: {t["caption"]}', style='Caption')
    tb = doc.add_table(rows=1, cols=len(df.columns))
    tb.style = 'Table Grid'
    for i, c in enumerate(df.columns):
        tb.rows[0].cells[i].text = str(c)
    for _, row in df.iterrows():
        cells_ = tb.add_row().cells
        for i, v in enumerate(row):
            cells_[i].text = '' if pd.isna(v) else str(v)
    for row in tb.rows:
        for c in row.cells:
            for pp in c.paragraphs:
                pp.paragraph_format.line_spacing = 1.0
                for rr in pp.runs: rr.font.size = Pt(9)
    doc.add_paragraph('Source: (add source – see References). Discussed in the text above.').runs[0].italic = True

def add_fig(f):
    doc.add_picture(f['path'], width=Cm(13.5))
    doc.paragraphs[-1].alignment = WD_ALIGN_PARAGRAPH.CENTER
    c = doc.add_paragraph(f'Figure {f["n"]}: {f["caption"]}', style='Caption')
    c.alignment = WD_ALIGN_PARAGRAPH.CENTER

guides = {
    'Introduction': 'Write the Introduction: background, objectives, study-area description and methodology (data sources, tools used – Python in Google Colab, with data from the course GitHub repository).',
    'Unit 1 – Physical Features': 'Discuss relief, climate, soils, vegetation and water. Explain every figure/table and how these features influence livelihoods, agriculture and settlement.',
    'Unit 2 – Administrative Structure, Population and Human Settlements': 'Outline administrative divisions, population size/density/growth/distribution, and settlement trends (urbanization, rural–urban migration, Imidugudu).',
    'Unit 3 – Socio-Economic Activities': 'Identify dominant sectors, development initiatives, investment zones, employment patterns; discuss challenges and opportunities for inclusive development.',
    'Unit 4 – Environmental Activities': 'Describe environmental programmes and challenges; link to Vision 2050, NST1/NST2 and the Green Growth and Climate Resilience Strategy.'}
order = ['Introduction', 'Unit 1 – Physical Features', 'Unit 2 – Administrative Structure, Population and Human Settlements',
         'Unit 3 – Socio-Economic Activities', 'Unit 4 – Environmental Activities']
for u_ in order:
    doc.add_heading(u_, 1)
    g_ = doc.add_paragraph('✍️ [WRITE YOUR DISCUSSION HERE] ' + guides[u_]); g_.runs[0].italic = True
    for t in [x for x in TABS if x['unit'] == u_]: add_table(t)
    for f in [x for x in FIGS if x['unit'] == u_]: add_fig(f)
    doc.add_paragraph()

doc.add_heading('Conclusion', 1)
doc.add_paragraph('✍️ [WRITE YOUR CONCLUSION AND RECOMMENDATIONS HERE]')
doc.add_heading('References', 1)
doc.add_paragraph('Dataset references generated from the analyses you ran. Add your report/policy/literature sources (NISR, REMA, RDB, RWB, MINALOC, MoE, Vision 2050, NST1/NST2, GGCRS…) using Mendeley, APA 7th edition. Check each reference before submitting.').runs[0].italic = True
for k in sorted(SOURCES): doc.add_paragraph(SOURCES[k])

fn = f'{OUT}/Group{GROUP_NUMBER}_{STUDY_NAME.replace(" ", "_")}_Report_Draft.docx'
doc.save(fn)
print('✅ Saved to Google Drive:', fn)
print(f'   {len(FIGS)} figures and {len(TABS)} tables inserted. In Word: right-click the Table of Contents → Update field.')
if DOWNLOAD_TO_PHONE:
    from google.colab import files
    files.download(fn)